# Baseball Broadcast Transcription Pipeline

This notebook transcribes old baseball radio broadcasts and produces a consolidated transcript for use in Retrosheet pitch sequence enrichment.

## Workflow
1. Configure game details
2. Verify GPU
3. Install dependencies
4. Mount Google Drive
5. Preprocess audio (upsample + enhance)
6. Split enhanced audio into chunks
7. Transcribe each chunk with Whisper
8. Merge transcripts into a single consolidated file
9. Preview transcript

## Requirements
- MP3 file of the broadcast uploaded to Google Drive
- Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

## Important
Always run Step 1 first — all other steps depend on the variables defined there.

---

## Step 1: Configuration

Set your game details here. **This is the only cell you need to edit for each new game.**
Run this cell before any other step.

In [ ]:
# -------------------------------------------------------------------
# CONFIGURATION — edit these values for each game
# -------------------------------------------------------------------

# Retrosheet game ID (e.g. NYA197409250)
GAME_ID = "NYN197409270"

# Name of the MP3 file in your Google Drive folder (include extension)
MP3_FILENAME = "1974 09-27 Pirates at Mets.mp3"

# Google Drive folder where your MP3 is stored and where output will be saved
DRIVE_FOLDER = "/content/drive/MyDrive/baseball_transcripts"

# Chunk size in seconds (1800 = 30 minutes)
CHUNK_SECONDS = 1800

# Whisper model to use
WHISPER_MODEL = "large-v3"

# Audio preprocessing settings
# Set SKIP_PREPROCESSING = True to use original audio without enhancement
SKIP_PREPROCESSING = False

# -------------------------------------------------------------------
# Imports and derived paths — no need to edit below this line
# -------------------------------------------------------------------
import os
import glob
import json
import subprocess
import time

MP3_PATH = os.path.join(DRIVE_FOLDER, MP3_FILENAME)
ENHANCED_WAV = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_enhanced.wav")
CHUNK_PREFIX = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_chunk")
MERGED_OUTPUT = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_FULL_TRANSCRIPT.json")
MERGED_TEXT_OUTPUT = os.path.join(DRIVE_FOLDER, f"{GAME_ID}_FULL_TRANSCRIPT.txt")

print(f"Game ID:        {GAME_ID}")
print(f"MP3 path:       {MP3_PATH}")
print(f"Enhanced audio: {ENHANCED_WAV}")
print(f"Chunk prefix:   {CHUNK_PREFIX}")
print(f"Merged output:  {MERGED_OUTPUT}")
print(f"Whisper model:  {WHISPER_MODEL}")
print(f"Preprocessing:  {'DISABLED' if SKIP_PREPROCESSING else 'ENABLED'}")
print("\n✅ Configuration loaded — ready to proceed")

## Step 2: Verify GPU

⚠️ **Before running this step**, make sure you have set the runtime to GPU:
1. Click **Runtime** in the top menu
2. Select **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save** — the session will restart
5. Re-run Step 1, then continue here

If `nvidia-smi: command not found` appears below, you are on a CPU runtime.

In [ ]:
import torch

!nvidia-smi

if torch.cuda.is_available():
    print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️  No GPU detected. Transcription will be much slower on CPU.")
    print("Go to Runtime → Change runtime type → T4 GPU")

## Step 3: Install Dependencies

In [ ]:
# Install Whisper and dependencies
!pip install -q -U openai-whisper

# Verify ffmpeg is available for audio processing
!which ffmpeg

print("\n✅ Dependencies installed")

## Step 4: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Guard: ensure Step 1 has been run
if 'MP3_PATH' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

# Verify the MP3 file exists
if not os.path.exists(MP3_PATH):
    raise FileNotFoundError(f"❌ MP3 file not found: {MP3_PATH}")

print(f"\n✅ Drive mounted and MP3 file found: {MP3_FILENAME}")

## Step 5: Preprocess Audio [NEW]

This step enhances the audio quality before transcription:
1. **Upsample** from 22.05 kHz to 16 kHz (Whisper's native sample rate)
2. **Apply noise reduction** to reduce tape hiss and crowd noise
3. **Enhance speech frequencies** (300-3000 Hz range)
4. **Normalize volume** for consistent levels
5. **Apply speech normalization** to improve dynamic range

This takes about 2-3 minutes for a 2-hour broadcast.

**Note:** The enhanced WAV file will be ~140 MB for a 2-hour game (vs ~17 MB for the original 32 kbps MP3). This is temporary and gets deleted when chunks are processed.

In [ ]:
# Guard: ensure Step 1 has been run
if 'MP3_PATH' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

if SKIP_PREPROCESSING:
    print("⏭️  Preprocessing disabled. Will use original MP3 for chunking.")
    AUDIO_FOR_CHUNKING = MP3_PATH
else:
    print("🔧 Preprocessing audio...")
    print("   This may take 2-3 minutes for a 2-hour broadcast.\n")
    
    # Check input file details
    probe_cmd = ['ffprobe', '-v', 'error', '-show_entries', 
                 'format=duration,bit_rate', '-show_entries', 
                 'stream=sample_rate,channels', '-of', 'json', MP3_PATH]
    probe_result = subprocess.run(probe_cmd, capture_output=True, text=True)
    print(f"Input file info:\n{probe_result.stdout}\n")
    
    # FFmpeg preprocessing pipeline optimized for 32 kbps / 22.05 kHz MP3s
    # This chain does:
    # 1. Convert to mono (saves processing time, speech is mono anyway)
    # 2. Resample to 16 kHz with high-quality SoX resampler
    # 3. High-pass filter at 300 Hz (removes rumble, tape noise)
    # 4. Low-pass filter at 3000 Hz (removes hiss, keeps speech)
    # 5. FFT denoise with moderate noise reduction
    # 6. Boost speech frequencies around 1 kHz (where consonants live)
    # 7. Normalize loudness to consistent level
    # 8. Speech normalization to compress dynamic range
    
    cmd = [
        'ffmpeg', '-y', '-i', MP3_PATH,
        '-ac', '1',  # Convert to mono
        '-ar', '16000',  # Resample to 16 kHz (Whisper's native rate)
        '-af', (
            'aresample=resampler=soxr:precision=28,'  # High-quality resampling
            'highpass=f=300,'  # Remove low rumble
            'lowpass=f=3000,'  # Remove high-frequency hiss
            'afftdn=nf=-20,'  # FFT denoise (nf=-20 is moderate)
            'equalizer=f=1000:width_type=o:width=2:g=6,'  # Boost speech range
            'loudnorm,'  # Normalize loudness
            'speechnorm=e=12.5:r=0.0001:l=1'  # Compress dynamic range for speech
        ),
        '-acodec', 'pcm_s16le',  # 16-bit PCM WAV
        ENHANCED_WAV
    ]
    
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start
    
    if result.returncode != 0:
        print(f"❌ FFmpeg failed:\n{result.stderr}")
        raise RuntimeError("Audio preprocessing failed")
    
    # Verify output
    if not os.path.exists(ENHANCED_WAV):
        raise FileNotFoundError(f"❌ Enhanced WAV not created: {ENHANCED_WAV}")
    
    size_mb = os.path.getsize(ENHANCED_WAV) / (1024 * 1024)
    
    print(f"✅ Audio preprocessing complete in {elapsed:.1f} seconds")
    print(f"   Enhanced WAV: {size_mb:.1f} MB")
    print(f"   Saved to: {ENHANCED_WAV}\n")
    
    AUDIO_FOR_CHUNKING = ENHANCED_WAV

print(f"Audio file for chunking: {AUDIO_FOR_CHUNKING}")

## Step 6: Split Audio into Chunks

Split the enhanced audio into 30-minute segments for efficient processing.

In [ ]:
# Guard: ensure Step 1 has been run
if 'AUDIO_FOR_CHUNKING' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 and Step 5 first.")

# Remove old chunks if they exist
old_chunks = glob.glob(f"{CHUNK_PREFIX}*.wav")
if old_chunks:
    print(f"🗑️  Removing {len(old_chunks)} old chunk(s)...")
    for chunk in old_chunks:
        os.remove(chunk)

# Split audio into chunks
print(f"✂️  Splitting audio into {CHUNK_SECONDS}-second chunks...\n")

cmd = [
    'ffmpeg', '-i', AUDIO_FOR_CHUNKING,
    '-f', 'segment',
    '-segment_time', str(CHUNK_SECONDS),
    '-c', 'copy',
    f"{CHUNK_PREFIX}_%03d.wav"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ FFmpeg failed:\n{result.stderr}")
    raise RuntimeError("Audio splitting failed")

chunks = sorted(glob.glob(f"{CHUNK_PREFIX}_*.wav"))
print(f"✅ Created {len(chunks)} chunk(s):\n")
for i, chunk in enumerate(chunks):
    size_mb = os.path.getsize(chunk) / (1024 * 1024)
    print(f"   Chunk {i}: {os.path.basename(chunk)} ({size_mb:.1f} MB)")

print(f"\n✅ Ready for transcription")

## Step 7: Transcribe Chunks with Whisper

Transcribe each chunk with optimized parameters for low-quality audio.

**Estimated time:** ~3-4 minutes per 30-minute chunk on T4 GPU (so ~25 minutes for a 2.5 hour game)

**New Whisper parameters for degraded audio:**
- `condition_on_previous_text=False` — prevents hallucinations when audio quality drops
- `temperature=0` — deterministic decoding for consistency
- `initial_prompt` — gives Whisper context about 1970s baseball terminology

In [ ]:
import whisper

# Guard: ensure Step 1 has been run
if 'CHUNK_PREFIX' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

# Load Whisper model
print(f"📥 Loading Whisper model: {WHISPER_MODEL}...")
model = whisper.load_model(WHISPER_MODEL)
print("✅ Model loaded\n")

# Find all chunks
chunks = sorted(glob.glob(f"{CHUNK_PREFIX}_*.wav"))
if not chunks:
    raise FileNotFoundError(f"❌ No chunks found with prefix: {CHUNK_PREFIX}_*.wav")

print(f"Found {len(chunks)} chunk(s) to transcribe\n")

# Initial prompt to give Whisper context about 1970s baseball
# This helps with terminology and proper nouns
BASEBALL_CONTEXT = (
    "1970s baseball radio broadcast. Play-by-play commentary with "
    "player names, team names, pitch calls, hits, strikes, balls, "
    "outs, innings, and game commentary."
)

# Transcribe each chunk
for i, chunk in enumerate(chunks):
    output_path = chunk.replace('.wav', '.json')
    
    # Skip if already transcribed
    if os.path.exists(output_path):
        print(f"⏭️  Chunk {i} already transcribed, skipping: {os.path.basename(chunk)}")
        continue
    
    print(f"🎙️  Transcribing chunk {i}/{len(chunks)-1}: {os.path.basename(chunk)}")
    start = time.time()
    
    # Transcribe with optimized parameters for degraded audio
    result = model.transcribe(
        chunk,
        language="en",
        # NEW: Disable conditioning on previous text to prevent hallucinations
        condition_on_previous_text=False,
        # NEW: Temperature 0 for deterministic, consistent output
        temperature=0,
        # NEW: Context about what we're transcribing
        initial_prompt=BASEBALL_CONTEXT,
        # Keep these for quality
        verbose=False
    )
    
    elapsed = time.time() - start
    
    # Save result as JSON
    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2)
    
    # Show progress
    seg_count = len(result['segments'])
    print(f"   ✅ Done in {elapsed:.1f}s ({seg_count} segments)\n")

print("\n🎉 All chunks transcribed!")

## Step 8: Merge Transcripts

Combine all chunk transcripts into a single consolidated file with continuous timestamps.

In [ ]:
# Guard: ensure Step 1 has been run
if 'MERGED_OUTPUT' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

# Find all transcript chunks
transcript_files = sorted(glob.glob(f"{CHUNK_PREFIX}_*.json"))
if not transcript_files:
    raise FileNotFoundError(f"❌ No transcripts found. Run Step 7 first.")

print(f"📄 Merging {len(transcript_files)} transcript(s)...\n")

merged_segments = []
time_offset = 0.0

for i, transcript_file in enumerate(transcript_files):
    with open(transcript_file) as f:
        data = json.load(f)
    
    # Adjust timestamps based on chunk position
    for seg in data['segments']:
        seg['start'] += time_offset
        seg['end'] += time_offset
        merged_segments.append(seg)
    
    # Update offset for next chunk (using actual audio duration)
    if data['segments']:
        time_offset = data['segments'][-1]['end']
    
    print(f"   Chunk {i}: {len(data['segments'])} segments")

# Create merged output
merged = {
    'game_id': GAME_ID,
    'total_segments': len(merged_segments),
    'duration_seconds': time_offset,
    'whisper_model': WHISPER_MODEL,
    'preprocessing_enabled': not SKIP_PREPROCESSING,
    'segments': merged_segments
}

# Save as JSON
with open(MERGED_OUTPUT, 'w') as f:
    json.dump(merged, f, indent=2)

# Also save as plain text for easy reading
with open(MERGED_TEXT_OUTPUT, 'w') as f:
    f.write(f"Game: {GAME_ID}\n")
    f.write(f"Duration: {time_offset/60:.1f} minutes\n")
    f.write(f"Segments: {len(merged_segments)}\n")
    f.write(f"Model: {WHISPER_MODEL}\n")
    f.write(f"Preprocessing: {'Yes' if not SKIP_PREPROCESSING else 'No'}\n")
    f.write("=" * 60 + "\n\n")
    
    for seg in merged_segments:
        start_min = int(seg['start'] // 60)
        start_sec = seg['start'] % 60
        f.write(f"[{start_min:02d}:{start_sec:05.2f}]  {seg['text'].strip()}\n")

print(f"\n✅ Merged transcript saved:")
print(f"   JSON: {MERGED_OUTPUT}")
print(f"   Text: {MERGED_TEXT_OUTPUT}")
print(f"\n📊 Stats:")
print(f"   Total duration: {time_offset/60:.1f} minutes")
print(f"   Total segments: {len(merged_segments)}")
print(f"   Avg segment length: {time_offset/len(merged_segments):.1f}s")

## Step 9: Preview Transcript

Preview a time window from the merged transcript.

In [ ]:
# Guard: ensure Step 1 has been run
if 'MERGED_OUTPUT' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

if not os.path.exists(MERGED_OUTPUT):
    print("❌ Merged transcript not found. Run Step 8 first.")
else:
    # Preview window in seconds — adjust as needed
    start_time = 0      # start of game
    end_time = 300      # first 5 minutes

    with open(MERGED_OUTPUT) as f:
        data = json.load(f)

    print(f"Game: {data['game_id']}")
    print(f"Model: {data['whisper_model']}")
    print(f"Preprocessing: {'Enabled' if data['preprocessing_enabled'] else 'Disabled'}")
    print(f"Previewing {start_time}s — {end_time}s\n")
    print("-" * 60)

    for seg in data["segments"]:
        if start_time <= seg["start"] <= end_time:
            start_min = int(seg['start'] // 60)
            start_sec = seg['start'] % 60
            print(f"[{start_min:02d}:{start_sec:05.2f}]  {seg['text'].strip()}")

## Step 10: Cleanup [Optional]

Remove intermediate files to save space:
- Enhanced WAV file (~140 MB for 2 hours)
- Chunk WAV files (~70 MB total for 2 hours)
- Chunk JSON files (~5 MB total for 2 hours)

**Only run this after you've verified the merged transcript is good!**

The final merged transcript files (JSON and TXT) will be preserved.

In [ ]:
# Guard: ensure Step 1 has been run
if 'CHUNK_PREFIX' not in dir():
    raise RuntimeError("❌ Configuration not loaded. Please run Step 1 first.")

print("🗑️  Cleaning up intermediate files...\n")

files_to_remove = []
total_size = 0

# Enhanced WAV
if os.path.exists(ENHANCED_WAV):
    size = os.path.getsize(ENHANCED_WAV)
    files_to_remove.append(ENHANCED_WAV)
    total_size += size
    print(f"   {os.path.basename(ENHANCED_WAV)}: {size/(1024*1024):.1f} MB")

# Chunk WAV files
chunk_wavs = glob.glob(f"{CHUNK_PREFIX}_*.wav")
for chunk in chunk_wavs:
    size = os.path.getsize(chunk)
    files_to_remove.append(chunk)
    total_size += size
print(f"   {len(chunk_wavs)} chunk WAV files: {sum(os.path.getsize(c) for c in chunk_wavs)/(1024*1024):.1f} MB")

# Chunk JSON files
chunk_jsons = glob.glob(f"{CHUNK_PREFIX}_*.json")
for chunk in chunk_jsons:
    size = os.path.getsize(chunk)
    files_to_remove.append(chunk)
    total_size += size
print(f"   {len(chunk_jsons)} chunk JSON files: {sum(os.path.getsize(c) for c in chunk_jsons)/(1024*1024):.1f} MB")

print(f"\nTotal space to reclaim: {total_size/(1024*1024):.1f} MB")
print(f"\n⚠️  This will delete {len(files_to_remove)} file(s).")
print("Your merged transcripts will be preserved:")
print(f"   - {MERGED_OUTPUT}")
print(f"   - {MERGED_TEXT_OUTPUT}")

# Uncomment the next block to actually delete files
# BE CAREFUL - this is permanent!

# for f in files_to_remove:
#     os.remove(f)
# print(f"\n✅ Cleanup complete. Reclaimed {total_size/(1024*1024):.1f} MB")

print("\n💡 To actually delete these files, uncomment the deletion code in this cell.")

---

## Notes & Improvements

### What's New in This Enhanced Version

**Audio Preprocessing (Step 5):**
- Upsamples from 22.05 kHz → 16 kHz (Whisper's native rate)
- High-pass filter at 300 Hz removes tape rumble
- Low-pass filter at 3000 Hz removes hiss
- FFT denoising reduces background noise
- EQ boost at 1 kHz enhances speech clarity
- Loudness normalization for consistent volume
- Speech normalization compresses dynamic range

**Optimized Whisper Parameters (Step 7):**
- `condition_on_previous_text=False` — prevents hallucinations in garbled sections
- `temperature=0` — deterministic decoding for consistency
- `initial_prompt` — gives context about 1970s baseball terminology

### Expected Results with 32 kbps Audio

Even with preprocessing, 32 kbps audio has fundamental limitations:

**Will likely improve:**
- ✅ General play-by-play flow (ball, strike, hit, out)
- ✅ Common player names (stars, regulars)
- ✅ Inning markers and score updates
- ✅ Clear announcer commentary

**Still challenging:**
- ⚠️ Less common player names (rookies, bench players)
- ⚠️ Rapid-fire pitch sequences during crowd noise
- ⚠️ Subtle commentary during loud moments
- ⚠️ Names/terms that sound similar ("Medich" vs "Manage")

### Comparing Results

To see if preprocessing helps, transcribe the same game twice:
1. Set `SKIP_PREPROCESSING = False` and run Steps 5-8
2. Set `SKIP_PREPROCESSING = True` and run Steps 5-8 with a different `GAME_ID`
3. Compare the two transcripts

### Further Tuning

If you're still getting too many errors, try:

**Less aggressive noise reduction:**
```python
'afftdn=nf=-25,'  # Instead of -20 (less aggressive)
```

**No speech threshold (experimental):**
```python
result = model.transcribe(
    chunk,
    language="en",
    condition_on_previous_text=False,
    temperature=0,
    initial_prompt=BASEBALL_CONTEXT,
    no_speech_threshold=0.7  # Default is 0.6, higher = less likely to transcribe noise
)
```

### If Audio Quality is Still the Bottleneck

Consider:
1. **Post-processing with LLM** — Use Claude or GPT-4 to clean up transcripts with baseball context
2. **Deepgram API** — Try their Nova-2 model on a single game (~$0.35 for 2 hours)
3. **Re-digitization** — If original tapes exist, this is the biggest win

### Post-Processing Example

You could add a Step 11 that uses the Claude API to clean up transcripts:

```python
# Pseudo-code for LLM post-processing
import anthropic

client = anthropic.Anthropic(api_key="your-key")

prompt = f"""
You are cleaning a transcript of a 1974 baseball broadcast.
Teams: Pittsburgh Pirates at New York Mets

Fix obvious mishearings while preserving authentic phrasing.
Mark uncertain words with [?].

Transcript:
{transcript_text}
"""
```

This is left as an exercise for the reader!